# 🚀 Deploying Used Car Price Prediction & Valuation System with FastAPI

This notebook demonstrates how to build and run a **production-style FastAPI application** to serve our champion **Tuned XGBoost Regressor** model. In a real-world scenario, web clients (front-ends, mobile apps) should not need to compute complex engineered features or scale columns before sending requests. Instead, they provide **raw car details**, and our API pipeline automatically validates, preprocesses, and evaluates the inputs on the server.

---

## Notebook Roadmap

| Section | Purpose | Key Technologies |
|---------|---------|------------------|
| **1** | FastAPI Architecture & Concepts | Markdown explanation |
| **2** | Environment Setup & Libraries | `pip`, `fastapi`, `uvicorn`, `nest_asyncio` |
| **3** | Model & Preprocessor Generation | `xgboost`, `scikit-learn`, `pickle` |
| **4** | Data Validation with Pydantic | `pydantic.BaseModel` |
| **5** | Preprocessing & Feature Engineering Pipeline | Custom Python Pipeline Class |
| **6** | FastAPI Server Implementation | `FastAPI`, Endpoints: `/`, `/health`, `/predict` |
| **7** | Running the Server Asynchronously | `nest_asyncio`, `threading` |
| **8** | Interactive API Testing | `requests` (GET, POST) |
| **9** | Swagger UI Guide | Interactive UI explanation |
| **10** | Production Deployment Considerations | Docker, Gunicorn, Monitoring, Caching |
| **11** | Final Summary | Accomplishment wrap-up |


## 1 · FastAPI Architecture & Concepts

### What is FastAPI?
FastAPI is a modern, fast (high-performance), web framework for building APIs with Python 3.8+ based on standard Python type hints. It sits on top of **Starlette** (for the web parts) and **Pydantic** (for the data parts), achieving speeds comparable to Node.js and Go.

### Architecture & Flow Diagram

When a user sends a POST request with raw car details, the API handles it in several distinct steps:

```
   [ Client Request ] (Raw JSON)
           │
           ▼
┌─────────────────────────┐
│ Pydantic Validation     │  <-- Checks types, ranges, missing fields
└──────────┬──────────────┘
           │ (Valid object)
           ▼
┌─────────────────────────┐
│ Preprocessing Pipeline  │  <-- Maps strings to label/one-hot encoders
└──────────┬──────────────┘
           │ (Mapped categorical fields)
           ▼
┌─────────────────────────┐
│ Feature Engineering     │  <-- Computes ratio columns & flag indicators
└──────────┬──────────────┘
           │ (81-feature full vector matches X_train)
           ▼
┌─────────────────────────┐
│ XGBoost Inference       │  <-- Predicts target price in Rupees
└──────────┬──────────────┘
           │ (Price prediction)
           ▼
┌─────────────────────────┐
│ Formatting & Response   │  <-- Creates Lakhs range, metadata, timestamp
└──────────┬──────────────┘
           │
           ▼
   [ Client Response ] (Clean JSON)
```

### Why Raw Inputs are Critical for Production
1. **Decoupled Architecture**: Front-end applications do not need to keep track of ML-specific logic like feature scaling, one-hot category positions, or mathematical ratios. They only need to know what a car is (oem, km, myear).
2. **Single Source of Truth**: The feature engineering equations (e.g. `KM_PER_YEAR = km / (car_age + 1)`) are defined in one place: the server. If we update our preprocessing logic or retrain our model, we do not need to push updates to client apps.
3. **Prevention of Model Poisoning**: Data validation (e.g., negative km, invalid year, empty model name) blocks bad inputs before they ever reach the model, preventing system crashes.


## 2 · Environment Setup & Libraries

We install the required libraries. Since FastAPI is an ASGI application, we need an ASGI server like **Uvicorn** to run it. We also use **nest-asyncio** to allow Uvicorn to run inside the Jupyter Notebook's event loop.


In [1]:
# Install FastAPI and related deployment dependencies
!pip install fastapi uvicorn nest-asyncio pydantic --quiet
print('✅ Deployment dependencies installed.')


✅ Deployment dependencies installed.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
import pydantic
import fastapi
import uvicorn
import nest_asyncio
import requests
import pickle
import time
import os
import logging
from pydantic import BaseModel, Field
from fastapi import FastAPI, HTTPException, status
from sklearn.preprocessing import LabelEncoder
from datetime import datetime

print(f'FastAPI version : {fastapi.__version__}')
print(f'Uvicorn version : {uvicorn.__version__}')
print(f'Pydantic version: {pydantic.__version__}')
print('All libraries loaded successfully.')


FastAPI version : 0.136.3
Uvicorn version : 0.49.0
Pydantic version: 2.11.9
All libraries loaded successfully.


## 3 · Model & Preprocessor Generation & Serialization

To make our deployment realistic, we first train our champion **Tuned XGBoost Regressor** using the feature engineered dataset. We also reconstruct the **LabelEncoders** from the clean dataset. Finally, we serialize both the model and the preprocessor metadata (including median values from the training set to fill default values) so our API can load them instantly.


In [3]:
# ── 3.1 Load Dataset and Train Champion Model ──
print('⌛ Loading feature engineered car data...')
df_eng = pd.read_csv('feature_engineered_car_data.csv')
X = df_eng.drop(columns=['listed_price', 'discountValue'])
y = df_eng['listed_price']

# Best hyperparameters discovered during Hyperparameter Tuning
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.9,
    min_child_weight=1,
    gamma=1.0,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42
)

start_time = time.time()
print('⌛ Training Tuned XGBoost model on all data...')
xgb_model.fit(X, y)
print(f'✅ Model trained successfully in {time.time() - start_time:.2f} seconds.')

# Save XGBoost Model in native JSON format
xgb_model.save_model('xgb_model.json')
print('💾 Saved model to "xgb_model.json".')


⌛ Loading feature engineered car data...
⌛ Training Tuned XGBoost model on all data...
✅ Model trained successfully in 10.16 seconds.
💾 Saved model to "xgb_model.json".


In [4]:
# ── 3.2 Reconstruct Label Encoders and Preprocessing Artifacts ──
print('⌛ Loading raw cleaned dataset to build encoders...')
df_raw = pd.read_csv('cars_data_clean.csv')

# Columns that were Label Encoded during preprocessing
high_cardinality = [
    'body', 'oem', 'model', 'variant', 'City', 
    'Engine Type', 'Gear Box', 'state', 'exterior_color', 'Fuel Suppy System'
]

label_encoders = {}
for col in high_cardinality:
    le = LabelEncoder()
    raw_col = df_raw[col].astype(str).str.lower().str.strip()
    le.fit(raw_col)
    label_encoders[col] = le
print(f'✅ LabelEncoders reconstructed for {len(high_cardinality)} columns.')

# Calculate median defaults from training set to fill columns the API user doesn't provide
defaults = {}
for col in X.columns:
    defaults[col] = X[col].median()
print(f'✅ Default values calculated for all {len(X.columns)} features.')

# Package preprocessor artifacts
preprocessor_artifacts = {
    'label_encoders': label_encoders,
    'defaults': defaults,
    'feature_names': list(X.columns),
    'median_km': df_eng['km'].median(),
    'median_car_age': df_eng['car_age'].median()
}

with open('preprocessor_artifacts.pkl', 'wb') as f:
    pickle.dump(preprocessor_artifacts, f)
print('💾 Preprocessor artifacts saved to "preprocessor_artifacts.pkl".')


⌛ Loading raw cleaned dataset to build encoders...
✅ LabelEncoders reconstructed for 10 columns.
✅ Default values calculated for all 81 features.
💾 Preprocessor artifacts saved to "preprocessor_artifacts.pkl".


## 4 · Data Input Validation with Pydantic

Pydantic uses Python type annotations to validate and parse data. We declare a schema for incoming requests using the `BaseModel` class. We specify descriptive types, validation constraints (e.g. year must be between 1980 and 2026, km must be non-negative), and default examples. This automatically generates schema metadata for the Swagger UI documentation.


In [5]:
class CarInputs(BaseModel):
    # Categorical strings
    oem: str = Field(..., example='Maruti', description='Original Equipment Manufacturer (Brand)')
    model: str = Field(..., example='Swift', description='Car Model name')
    fuel: str = Field(..., example='Petrol', description='Fuel type (Petrol, Diesel, CNG, LPG, Electric)')
    transmission: str = Field(..., example='Manual', description='Transmission type (Manual, Automatic)')
    owner_type: str = Field(..., example='First', description='Ownership history (First, Second, Third, Fourth, Unregistered Car)')
    
    # Numerical inputs
    myear: int = Field(..., example=2018, ge=1980, le=2026, description='Manufacturing Year')
    km: float = Field(..., example=50000.0, ge=0.0, description='Total kilometers driven')
    engine_cc: float = Field(..., example=1197.0, ge=0.0, description='Engine capacity in cc')
    max_power_bhp: float = Field(..., example=83.0, ge=0.0, description='Maximum power in bhp')
    max_torque_nm: float = Field(..., example=113.0, ge=0.0, description='Maximum torque in Nm')
    kerb_weight: float = Field(..., example=960.0, ge=0.0, description='Weight of the vehicle empty')
    length: float = Field(..., example=3840.0, ge=0.0, description='Vehicle length in mm')
    width: float = Field(..., example=1735.0, ge=0.0, description='Vehicle width in mm')
    height: float = Field(..., example=1530.0, ge=0.0, description='Vehicle height in mm')
    no_of_cylinder: int = Field(..., example=4, ge=1, description='Number of engine cylinders')

print('✅ Pydantic schema class defined successfully.')


✅ Pydantic schema class defined successfully.


## 5 · Preprocessing & Feature Engineering Pipeline

We create a preprocessing function that transforms a raw `CarInputs` model into a model-ready 2D array. The pipeline runs the following server-side steps:
1. **Initialize Feature Vector**: Creates a dictionary populated with training-set defaults for all 81 features.
2. **Map Numerical Inputs**: Assigns raw inputs (e.g. `km`, `max_power_bhp` -> `Max Power Delivered`) directly.
3. **Encode Categorical Strings**: Matches `oem` and `model` to their LabelEncoder categories, converting them to integers. If an unseen brand or model is provided, it falls back to a default value (e.g., 0).
4. **Map One-Hot Encoders**: Sets binary indicators (e.g., `fuel_diesel`, `transmission_manual`, `owner_type_*`) to `1` or `0` based on the string values.
5. **Perform Feature Engineering**: Computes the exact ratio and flag indicators created during the Feature Engineering phase:
   - `car_age = 2026 - myear`
   - `KM_PER_YEAR = km / (car_age + 1)`
   - `POWER_TO_WEIGHT = max_power_bhp / (kerb_weight + 1)`
   - `TORQUE_TO_POWER = max_torque_nm / (max_power_bhp + 1)`
   - `ENGINE_PER_CYLINDER = max_power_bhp / (no_of_cylinder + 1)`
   - `CAR_VOLUME = (length * width * height) / 1,000,000`
   - Brand checks for `LUXURY_BRAND_FLAG` and `PREMIUM_BRAND_FLAG` using OEM lists.
   - High-mileage and old-car binary flags.
6. **Align Columns**: Conforms the feature keys exactly to the original training list order (`preprocessor_artifacts['feature_names']`) and reshapes it to shape `(1, 81)` for XGBoost inference.


In [6]:
def preprocess_raw_inputs(inputs: CarInputs, artifacts: dict) -> pd.DataFrame:
    """
    Transforms raw validation fields to the engineered 81-feature structure.
    """
    # Start with training medians as defaults for unprovided features (e.g. Variant, City, state, etc.)
    features = artifacts['defaults'].copy()
    
    # Direct Numerical Mapping
    features['km'] = inputs.km
    features['Length'] = inputs.length
    features['Width'] = inputs.width
    features['Height'] = inputs.height
    features['Kerb Weight'] = inputs.kerb_weight
    features['No of Cylinder'] = float(inputs.no_of_cylinder)
    features['Max Power Delivered'] = inputs.max_power_bhp
    features['Max Torque Delivered'] = inputs.max_torque_nm
    
    # Categorical Label Encoding Mapping
    clean_oem = inputs.oem.lower().strip()
    clean_model = inputs.model.lower().strip()
    
    le_oem = artifacts['label_encoders']['oem']
    le_model = artifacts['label_encoders']['model']
    
    # Fallback to default index if unseen label is provided
    features['oem'] = le_oem.transform([clean_oem])[0] if clean_oem in le_oem.classes_ else 0
    features['model'] = le_model.transform([clean_model])[0] if clean_model in le_model.classes_ else 0
    
    # One-Hot Encoding Mapping - Transmission
    features['transmission_manual'] = 1 if inputs.transmission.lower().strip() == 'manual' else 0
    
    # One-Hot Encoding Mapping - Fuel
    clean_fuel = inputs.fuel.lower().strip()
    features['fuel_diesel'] = 1 if clean_fuel == 'diesel' else 0
    features['fuel_electric'] = 1 if clean_fuel == 'electric' else 0
    features['fuel_lpg'] = 1 if clean_fuel == 'lpg' else 0
    features['fuel_petrol'] = 1 if clean_fuel == 'petrol' else 0
    # Note: CNG is the dropped first category, so if fuel == 'cng', all above remain 0
    
    # One-Hot Encoding Mapping - Owner Type
    clean_owner = inputs.owner_type.lower().strip()
    features['owner_type_first'] = 0
    features['owner_type_second'] = 0
    features['owner_type_third'] = 0
    features['owner_type_fourth'] = 0
    features['owner_type_unregistered car'] = 0
    
    if clean_owner == 'first':
        features['owner_type_first'] = 1
    elif clean_owner == 'second':
        features['owner_type_second'] = 1
    elif clean_owner == 'third':
        features['owner_type_third'] = 1
    elif clean_owner == 'fourth':
        features['owner_type_fourth'] = 1
    elif clean_owner in ['unregistered', 'unregistered car']:
        features['owner_type_unregistered car'] = 1
        
    # ── Feature Engineering calculations ──
    CURRENT_YEAR = 2026
    car_age = CURRENT_YEAR - inputs.myear
    features['car_age'] = float(car_age)
    features['KM_PER_YEAR'] = inputs.km / (car_age + 1.0)
    features['POWER_TO_WEIGHT'] = inputs.max_power_bhp / (inputs.kerb_weight + 1.0)
    features['TORQUE_TO_POWER'] = inputs.max_torque_nm / (inputs.max_power_bhp + 1.0)
    features['ENGINE_PER_CYLINDER'] = inputs.max_power_bhp / (float(inputs.no_of_cylinder) + 1.0)
    features['CAR_VOLUME'] = (inputs.length * inputs.width * inputs.height) / 1e6
    
    # Brand flags
    LUXURY_BRANDS = ['bmw', 'audi', 'mercedes-benz', 'jaguar', 'volvo', 'land rover',
                     'porsche', 'bentley', 'lamborghini', 'maserati', 'rolls-royce']
    PREMIUM_BRANDS = ['honda', 'toyota', 'hyundai', 'volkswagen', 'skoda', 'kia']
    
    features['LUXURY_BRAND_FLAG'] = 1 if clean_oem in LUXURY_BRANDS else 0
    features['PREMIUM_BRAND_FLAG'] = 1 if clean_oem in PREMIUM_BRANDS else 0
    
    # Median checks
    features['HIGH_MILEAGE_FLAG'] = 1 if inputs.km > artifacts['median_km'] else 0
    features['OLD_CAR_FLAG'] = 1 if car_age > artifacts['median_car_age'] else 0
    
    # Convert to DataFrame and align columns exactly with training data
    df_vec = pd.DataFrame([features])
    df_vec = df_vec[artifacts['feature_names']]
    return df_vec

print('✅ Preprocessing pipeline function created.')


✅ Preprocessing pipeline function created.


## 6 · FastAPI Server Implementation

We initialize our FastAPI application. We load our model and preprocessor artifacts during startup. We write robust endpoints, adding request exception handling, predictive interval calculations, and standard logging.


In [7]:
app = FastAPI(
    title="Used Car Price Valuation API",
    description="Production REST API for Used Car Price Prediction & Valuation System.",
    version="1.0.0"
)

# Global states for artifacts
model = None
artifacts = None

@app.on_event("startup")
def load_resources():
    """Startup task to load machine learning artifacts from disk."""
    global model, artifacts
    try:
        # Load XGBoost model
        model = xgb.XGBRegressor()
        model.load_model('xgb_model.json')
        
        # Load Preprocessor metadata
        with open('preprocessor_artifacts.pkl', 'rb') as f:
            artifacts = pickle.load(f)
            
        print('🚀 Server startup: XGBoost model & Preprocessor loaded successfully.')
    except Exception as e:
        print(f'❌ Startup failure: {e}')
        raise RuntimeError(f'Resource loading failed: {e}')

@app.get("/", tags=["Status"])
def read_root():
    """API Root status endpoint."""
    return {
        "api_name": "Used Car Price Valuation API",
        "status": "online",
        "version": "1.0.0",
        "documentation_url": "/docs"
    }

@app.get("/health", tags=["Status"])
def health_check():
    """System health and dependency check."""
    is_healthy = (model is not None) and (artifacts is not None)
    return {
        "status": "healthy" if is_healthy else "unhealthy",
        "model_loaded": model is not None,
        "preprocessor_loaded": artifacts is not None,
        "timestamp": datetime.now().isoformat()
    }

@app.post("/predict", tags=["Valuation"], status_code=status.HTTP_200_OK)
def predict_car_price(payload: CarInputs):
    """
    Endpoint to predict used car valuation based on raw details.
    """
    if model is None or artifacts is None:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Server model is not loaded yet."
        )
        
    try:
        # 1. Transform raw payload to 81-feature array
        df_input = preprocess_raw_inputs(payload, artifacts)
        
        # 2. Inference
        pred_raw = float(model.predict(df_input)[0])
        
        # Post-processing: Ensure prediction is not negative
        predicted_price = max(0.0, round(pred_raw))
        
        # 3. Calculate price range (Valuation boundaries: +/- 5%)
        lower_bound = predicted_price * 0.95
        upper_bound = predicted_price * 1.05
        
        # Format to Lakhs (e.g. ₹7.8L - ₹8.7L)
        lower_lakhs = lower_bound / 100000
        upper_lakhs = upper_bound / 100000
        price_range = f"₹{lower_lakhs:.1f}L - ₹{upper_lakhs:.1f}L"
        
        # Response body
        return {
            "predicted_price": predicted_price,
            "price_range": price_range,
            "model_version": "1.0.0",
            "prediction_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
    except Exception as e:
        # Request logging for diagnostic monitoring
        logging.error(f'Valuation inference error: {e}')
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Inference valuation error occurred: {str(e)}"
        )

print('✅ FastAPI application structure and endpoints implemented.')


✅ FastAPI application structure and endpoints implemented.


C:\Users\admin\AppData\Local\Temp\ipykernel_11368\3501845687.py:11: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


## 7 · Jupyter Integration (nest-asyncio)

Jupyter Notebooks run inside an active asyncio event loop. Attempting to start another ASGI server inside the notebook normally causes a loop collision crash (`RuntimeError: This event loop is already running`).

We use **`nest-asyncio`** to patch the event loop, allowing our local web server to execute concurrently in the background.


In [8]:
# Patch event loop
nest_asyncio.apply()
print('✅ nest-asyncio applied. Jupyter event loop patched.')


✅ nest-asyncio applied. Jupyter event loop patched.


## 8 · Running the FastAPI Server

We start Uvicorn in a **background thread**. This ensures the server does not block notebook execution, so we can write client-side HTTP calls directly in subsequent cells.


In [9]:
import threading

def start_uvicorn():
    # Runs FastAPI server on localhost:8000
    # We set log_level to warning to reduce console clutter inside Jupyter
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

# Create and launch server thread
server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
print('⌛ Starting local FastAPI server in background thread...')
time.sleep(3)  # Give uvicorn a brief moment to initialize
print('🚀 Local server should now be running on http://127.0.0.1:8000')


⌛ Starting local FastAPI server in background thread...
🚀 Server startup: XGBoost model & Preprocessor loaded successfully.
🚀 Local server should now be running on http://127.0.0.1:8000


## 9 · Interactive API Testing

Now we write client requests using Python's `requests` library to verify the endpoints. We test GET `/health`, successful predictions, and validation errors (422 Unprocessable Entity) on bad payloads.


In [10]:
# ── 9.1 Call Root and Health Checks ──
BASE_URL = 'http://127.0.0.1:8000'

# Root status
resp_root = requests.get(BASE_URL)
print('Root endpoint check:')
print(resp_root.json())

# Health status
resp_health = requests.get(f'{BASE_URL}/health')
print('\nHealth endpoint check:')
print(resp_health.json())


Root endpoint check:
{'api_name': 'Used Car Price Valuation API', 'status': 'online', 'version': '1.0.0', 'documentation_url': '/docs'}

Health endpoint check:
{'status': 'healthy', 'model_loaded': True, 'preprocessor_loaded': True, 'timestamp': '2026-06-21T17:20:36.114887'}


In [11]:
# ── 9.2 Call Prediction with a Valid Sample ──
valid_payload = {
    "oem": "Maruti",
    "model": "Swift",
    "fuel": "Petrol",
    "transmission": "Manual",
    "owner_type": "First",
    "myear": 2018,
    "km": 50000.0,
    "engine_cc": 1197.0,
    "max_power_bhp": 83.0,
    "max_torque_nm": 113.0,
    "kerb_weight": 960.0,
    "length": 3840.0,
    "width": 1735.0,
    "height": 1530.0,
    "no_of_cylinder": 4
}

resp_pred = requests.post(f'{BASE_URL}/predict', json=valid_payload)
print('Valid Prediction Response Status Code:', resp_pred.status_code)
print('JSON Output:')
import json
print(json.dumps(resp_pred.json(), indent=4, ensure_ascii=False))


Valid Prediction Response Status Code: 200
JSON Output:
{
    "predicted_price": 545633,
    "price_range": "₹5.2L - ₹5.7L",
    "model_version": "1.0.0",
    "prediction_timestamp": "2026-06-21 17:20:43"
}


In [12]:
# ── 9.3 Call Prediction with an Invalid Sample (Triggers Pydantic validation) ──
invalid_payload = {
    "oem": "Maruti",
    "model": "Swift",
    "fuel": "Petrol",
    "transmission": "Manual",
    "owner_type": "First",
    "myear": 2030,  # <-- Invalid Year (max constraint is 2026)
    "km": -10.0,     # <-- Invalid Km (minimum constraint is 0.0)
    "engine_cc": 1197.0,
    "max_power_bhp": 83.0,
    "max_torque_nm": 113.0,
    "kerb_weight": 960.0,
    "length": 3840.0,
    "width": 1735.0,
    "height": 1530.0,
    "no_of_cylinder": 4
}

resp_invalid = requests.post(f'{BASE_URL}/predict', json=invalid_payload)
print('Invalid Request Status Code:', resp_invalid.status_code)
print('Validation Error JSON:')
print(json.dumps(resp_invalid.json(), indent=4))


Invalid Request Status Code: 422
Validation Error JSON:
{
    "detail": [
        {
            "type": "less_than_equal",
            "loc": [
                "body",
                "myear"
            ],
            "msg": "Input should be less than or equal to 2026",
            "input": 2030,
            "ctx": {
                "le": 2026
            }
        },
        {
            "type": "greater_than_equal",
            "loc": [
                "body",
                "km"
            ],
            "msg": "Input should be greater than or equal to 0",
            "input": -10.0,
            "ctx": {
                "ge": 0.0
            }
        }
    ]
}


## 10 · Swagger UI Interactive Testing Guide

FastAPI automatically parses Pydantic models to generate interactive API documentation page using **Swagger UI**. To access it:

1. Open your browser and navigate to: **[http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)**
2. Locate the `POST /predict` route block.
3. Click **"Try it out"** on the right side.
4. Modify the request body JSON parameter values (e.g. change brand, year, mileage).
5. Click **"Execute"**.
6. Scroll down to see the real-time valuation result, response time, and curl command headers.

> [!NOTE]
> The Swagger page is generated on the fly directly from the Pydantic type annotation schemas and validation parameters.


## 11 · Production Deployment Considerations

Deploying a prototype API locally inside a Jupyter Notebook is fine for testing, but in a production environment, you should consider the following best practices:

### 1. Scaling ASGI Web Servers
- Uvicorn is single-threaded. In production, wrap it in a process manager like **Gunicorn** to handle concurrency across multiple CPU cores.
- Production deployment command: `gunicorn -w 4 -k uvicorn.workers.UvicornWorker main:app` (where `-w 4` runs 4 parallel worker instances).

### 2. Containerization (Docker)
- Containerize the application and dependencies to ensure portability across staging and cloud environments.
- Example production `Dockerfile`:
  ```dockerfile
  FROM python:3.10-slim
  WORKDIR /app
  COPY requirements.txt .
  RUN pip install --no-cache-dir -r requirements.txt
  COPY xgb_model.json .
  COPY preprocessor_artifacts.pkl .
  COPY main.py .
  EXPOSE 8000
  CMD ["gunicorn", "-w", "4", "-k", "uvicorn.workers.UvicornWorker", "main:app", "--bind", "0.0.0.0:8000"]
  ```

### 3. Monitoring & Performance
- **Health Checking**: Connect `/health` checks to orchestration services like Kubernetes liveness/readiness probes.
- **Caching**: Implement a **Redis** cache layer in front of `/predict` to store predictions for identical raw specs (e.g., highly popular vehicle specifications).
- **Metrics**: Implement Prometheus middleware to track model inference latency, API requests per second, and 500-level HTTP error rates.


## 12 · Summary & Next Steps

In this notebook, we accomplished:
- Trained and serialized our **Tuned XGBoost Regressor** Champion model.
- Saved statistical preprocessing properties (Label encoders, medians) to support **server-side feature engineering**.
- Wrote Pydantic data schemas for **strict type validation**.
- Implemented a fully functional **FastAPI application** with status checks and model prediction logic.
- Ran and verified the server asynchronously inside Jupyter.

You have completed the entire Used Car Price Prediction & Valuation pipeline. This deployment model is ready to be loaded into a Docker container and deployed to production!
